In [8]:
#!/usr/bin/env python3
"""
MEXC OHLCV Downloader  ― 1 分足なし / JST 対応版
"""

from __future__ import annotations

import os
import time
import math
from datetime import datetime, timezone, timedelta
from typing import Dict, Tuple

import ccxt
import pandas as pd
from tqdm import tqdm

# ─────────────────────────────
#  設定
# ─────────────────────────────
CONFIG: Dict[str, Dict[str, Tuple[str, str]]] = {
    "SOL/USDC": {
        "5m":  ("2025-04-20", "2025-07-19"),
        "15m": ("2025-01-20", "2025-07-19"),
        "1h":  ("2024-07-20", "2025-07-19"),
    }
}
OUTPUT_DIR   = "ohlcv_data"
FILE_FMT     = "csv"          # csv | parquet
MAX_RETRIES  = 5
SLEEP_JITTER = 0.05
JST          = timezone(timedelta(hours=9))  # UTC+9

# ─────────────────────────────
#  ユーティリティ
# ─────────────────────────────
def iso_to_ms(date_str: str) -> int:
    """
    'YYYY-MM-DD' または ISO-8601 文字列を
    「JST 0:00」として Epoch ミリ秒 (UTC) に変換
    """
    if len(date_str) == 10:                   # 例: '2025-04-20'
        dt = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=JST)
    else:                                     # ISO 文字列 (タイムゾーン省略可)
        dt = datetime.fromisoformat(date_str)
        if dt.tzinfo is None:
            dt = dt.replace(tzinfo=JST)
    return int(dt.timestamp() * 1000)         # timestamp() は UTC 秒

def fetch_ohlcv_range(
    exchange: ccxt.Exchange,
    symbol: str,
    timeframe: str,
    since_ms: int,
    until_ms: int,
) -> pd.DataFrame:
    """since_ms から until_ms までローソク足を取得し DataFrame で返す (datetime=JST)"""
    all_rows = []
    limit  = 1000
    tf_ms  = exchange.parse_timeframe(timeframe) * 1000
    total  = math.ceil((until_ms - since_ms) / tf_ms)

    pbar = tqdm(total=total, unit="candle", desc=f"{symbol} {timeframe}")

    while since_ms < until_ms:
        # ① API 呼び出し（リトライ付き）
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                batch = exchange.fetch_ohlcv(symbol, timeframe, since=since_ms, limit=limit)
                break
            except Exception as e:
                if attempt == MAX_RETRIES:
                    raise
                wait = exchange.rateLimit / 1000 + attempt * 0.5
                print(f"⚠️  {symbol} {timeframe}: {e} (retry {attempt}/{MAX_RETRIES}, wait {wait}s)")
                time.sleep(wait)

        if not batch:  # 空レスポンス
            break

        # ③ 重複除去 & 範囲制限
        batch = [c for c in batch if since_ms <= c[0] < until_ms]
        if batch and all_rows and batch[0][0] == all_rows[-1][0]:
            batch = batch[1:]
        all_rows.extend(batch)
        pbar.update(len(batch))

        # ④ 次の since
        last_ts = batch[-1][0] if batch else since_ms
        since_ms = last_ts + tf_ms

        # ⑤ レート制限待機
        time.sleep(exchange.rateLimit / 1000 + SLEEP_JITTER)

    pbar.close()

    columns = ["timestamp", "open", "high", "low", "close", "volume"]
    df = pd.DataFrame(all_rows, columns=columns)
    # ↓ UTC → JST へタイムゾーン変換
    df["datetime"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert(JST)
    return df[["datetime", "open", "high", "low", "close", "volume"]]

def save_df(df: pd.DataFrame, path: str):
    """DataFrame を CSV または Parquet で保存 (datetime は JST)"""
    if FILE_FMT == "parquet":
        df.to_parquet(path, index=False)
    else:
        df.to_csv(path, index=False)

# ─────────────────────────────
#  メイン
# ─────────────────────────────
def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    exchange = ccxt.mexc({"enableRateLimit": True})

    for symbol, tf_cfg in CONFIG.items():
        symbol_slug = symbol.replace("/", "").lower()

        for tf, (start, end) in tf_cfg.items():
            since_ms = iso_to_ms(start)
            until_ms = iso_to_ms(end)

            outfile = os.path.join(
                OUTPUT_DIR,
                f"{symbol_slug}_{start.replace('-', '')}_{end.replace('-', '')}_{tf}.{FILE_FMT}"
            )

            # 既存ファイルがあれば再開
            if os.path.exists(outfile):
                print(f"🔄 Resuming {outfile}")
                if FILE_FMT == "parquet":
                    existing = pd.read_parquet(outfile)
                else:
                    existing = pd.read_csv(outfile, parse_dates=["datetime"])

                if not existing.empty:
                    last_ts_ms = int(existing["datetime"].iloc[-1].timestamp() * 1000)
                    since_ms = last_ts_ms + exchange.parse_timeframe(tf) * 1000

            jst_start_str = datetime.fromtimestamp(since_ms / 1000, JST).strftime("%Y-%m-%d")
            print(f"▶ Fetching {symbol} {tf}: {jst_start_str} JST → {end}")

            df = fetch_ohlcv_range(exchange, symbol, tf, since_ms, until_ms)

            if df.empty:
                print(f"⚠️  No data for {symbol} {tf}")
                continue

            save_df(df, outfile)
            print(f"✅ Saved {len(df):,} rows → {outfile}\n")

    print("🎉 All downloads complete.")

if __name__ == "__main__":
    main()


▶ Fetching SOL/USDC 5m: 2025-04-20 JST → 2025-07-19


SOL/USDC 5m: 100%|██████████| 25920/25920 [00:11<00:00, 2299.57candle/s]


✅ Saved 25,920 rows → ohlcv_data\solusdc_20250420_20250719_5m.csv

▶ Fetching SOL/USDC 15m: 2025-01-20 JST → 2025-07-19


SOL/USDC 15m: 100%|██████████| 17280/17280 [00:05<00:00, 3205.83candle/s]


✅ Saved 17,280 rows → ohlcv_data\solusdc_20250120_20250719_15m.csv

▶ Fetching SOL/USDC 1h: 2024-07-20 JST → 2025-07-19


SOL/USDC 1h: 100%|██████████| 8736/8736 [00:02<00:00, 3203.08candle/s]


✅ Saved 8,736 rows → ohlcv_data\solusdc_20240720_20250719_1h.csv

🎉 All downloads complete.
